In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session


/kaggle/input/train.csv
/kaggle/input/train.csv.zip
/kaggle/input/sample_submission.csv
/kaggle/input/sample_submission.csv.zip
/kaggle/input/test.csv
/kaggle/input/test.csv.zip
/kaggle/input/description.md
/kaggle/input/ventilator-pressure-prediction/train.csv
/kaggle/input/ventilator-pressure-prediction/train.csv.zip
/kaggle/input/ventilator-pressure-prediction/sample_submission.csv
/kaggle/input/ventilator-pressure-prediction/sample_submission.csv.zip
/kaggle/input/ventilator-pressure-prediction/test.csv
/kaggle/input/ventilator-pressure-prediction/test.csv.zip
/kaggle/input/ventilator-pressure-prediction/description.md


In [2]:
# libraries
import pandas as pd
from tqdm.auto import tqdm
tqdm.pandas()
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
import os
import numpy as np
import time
import itertools
import tensorflow as tf
import tensorflow.keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import *
from tensorflow.keras.wrappers.scikit_learn import KerasRegressor
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import *
from sklearn.model_selection import train_test_split
from sklearn.ensemble import VotingRegressor

import random
random.seed(7)
tf.random.set_seed(7)


2026-01-07 06:17:52.114055: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767766672.127499      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767766672.131589      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-01-07 06:17:52.147912: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

ModuleNotFoundError: No module named 'tensorflow.keras.wrappers.scikit_learn'

In [3]:
directory = '../input/ventilator-pressure-prediction'
train = pd.read_csv(os.path.join(directory, 'train.csv'))
test = pd.read_csv(os.path.join(directory, 'test.csv'))
sub = pd.read_csv(os.path.join(directory, 'sample_submission.csv'))


In [4]:
train[['R', 'C', 'time_step', 'u_in', 'u_out']].values


array([[5.00000000e+00, 1.00000000e+01, 0.00000000e+00, 4.17441921e+00,
        0.00000000e+00],
       [5.00000000e+00, 1.00000000e+01, 3.38120461e-02, 7.05014900e+00,
        0.00000000e+00],
       [5.00000000e+00, 1.00000000e+01, 6.74965382e-02, 7.56493080e+00,
        0.00000000e+00],
       ...,
       [5.00000000e+01, 1.00000000e+01, 2.65599585e+00, 4.98471146e+00,
        1.00000000e+00],
       [5.00000000e+01, 1.00000000e+01, 2.69013524e+00, 4.98711058e+00,
        1.00000000e+00],
       [5.00000000e+01, 1.00000000e+01, 2.72509122e+00, 4.98917750e+00,
        1.00000000e+00]])

In [5]:
X = train[['R', 'C', 'time_step', 'u_in', 'u_out']].values
Y = train.pressure.values

sample_length = 80
input_dataset = tf.keras.preprocessing.timeseries_dataset_from_array(
  X, None, sequence_length=sample_length, sequence_stride=sample_length)
target_dataset = tf.keras.preprocessing.timeseries_dataset_from_array(
  Y, None, sequence_length=sample_length, sequence_stride=sample_length)

for batch in zip(input_dataset, target_dataset):
  inputs, targets = batch
  assert np.array_equal(inputs[0], X[:sample_length])

  # second sample equals output timestamps 20-40
  assert np.array_equal(targets[1], Y[sample_length:2*sample_length])
  break


2026-01-07 06:18:02.016247: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1767766682.016708      11 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 46866 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:85:00.0, compute capability: 8.6


In [6]:
import time
import tensorflow.keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import *
from tensorflow.keras.wrappers.scikit_learn import KerasRegressor
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import *
from sklearn.model_selection import train_test_split
from sklearn.ensemble import VotingRegressor


ModuleNotFoundError: No module named 'tensorflow.keras.wrappers.scikit_learn'

In [7]:
train.shape


(5432400, 8)

In [8]:
def lstm_model():
    # create model
    model = Sequential()
    model.add(Input(shape = (inputs.shape[1], inputs.shape[2])))
    model.add(LSTM(320, return_sequences=True))
    model.add(LSTM(320, return_sequences=True))
    model.add(Dense(160, activation = 'relu'))
    #model.add(Dense(80, activation = 'relu'))
    model.add(Dense(1, kernel_initializer='normal'))
    model.compile(loss='mae', optimizer='adam', metrics = [tensorflow.keras.metrics.RootMeanSquaredError()])
    return model


In [9]:
def bi_lstm_model():
    # create model
    model = Sequential()
    model.add(Input(shape = (inputs.shape[1], inputs.shape[2])))
    model.add(Bidirectional(LSTM(640, return_sequences=True)))
    model.add(Bidirectional(LSTM(320, return_sequences=True)))
    model.add(LSTM(320, return_sequences=True))
    model.add(LSTM(320, return_sequences=True))
    model.add(Dense(320, activation = 'relu'))
    model.add(Dense(160, activation = 'relu'))
    #model.add(Dense(80, activation = 'relu'))
    model.add(Dense(1, kernel_initializer='normal'))
    model.compile(loss='mae', optimizer='adam', metrics = [tensorflow.keras.metrics.RootMeanSquaredError()])
    return model


In [10]:
def cnn_lstm_model():
    # create model

    model = Sequential()
    model.add(Input(shape = ( inputs.shape[1], inputs.shape[2])))
    model.add(Conv1D(filters = 320, kernel_size = 3, strides = 1, padding = 'causal', activation = "relu" ))
    model.add(LSTM(320, return_sequences=True, activation = 'tanh'))
    model.add(LSTM(320, return_sequences=True, activation = 'tanh'))
    model.add(Dense(160, activation = 'relu'))
    #model.add(Dense(80, activation = 'relu'))
    model.add(Dense(1, kernel_initializer='normal'))
    model.compile(loss='mae', optimizer = tensorflow.keras.optimizers.Adam(), metrics = [tensorflow.keras.metrics.RootMeanSquaredError()])
    return model


In [11]:
# define the stages of the pipeline
pipeline = Pipeline(steps= [('model', KerasRegressor(build_fn=cnn_lstm_model, epochs=500, batch_size = 512, verbose=1))])

# fit the pipeline model with the training data                            
pipeline.fit(inputs, targets)


NameError: name 'KerasRegressor' is not defined

In [12]:
4024000/80


50300.0

In [13]:
sub_array = pipeline.predict(test[['R', 'C', 'time_step', 'u_in', 'u_out']].to_numpy().reshape(50300, 80, 5))
#list(itertools.chain(*sub_array))


NameError: name 'pipeline' is not defined

In [14]:
sub_array =list(itertools.chain(*sub_array))


NameError: name 'sub_array' is not defined

In [15]:
#sub_array = pipeline.predict(test[['R', 'C', 'time_step', 'u_in', 'u_out']])


In [16]:
pd.DataFrame({'id':test.id, 'pressure':sub_array}).to_csv('mark_5.csv', index=False)


NameError: name 'sub_array' is not defined